# 03 - Hidden Markov Model: Market Regime Detection

## Project: Crypto Market Regime Detection using Hidden Markov Models

This notebook applies a Gaussian HMM to the standardized features to discover hidden market regimes (bull, bear, lateral) in Bitcoin's price history.

**Key concepts:**
- Transition matrix: how the market moves between regimes
- Emission distributions: how each regime "looks" in terms of features
- Viterbi decoding: finding the most probable sequence of regimes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from hmmlearn.hmm import GaussianHMM

plt.style.use('seaborn-v0_8-darkgrid')
btc_scaled = pd.read_csv('../data/raw/btc_features_scaled.csv', index_col='Date', parse_dates=True)
# scaler = joblib.load('../data/raw/feature_scaler.pkl')
btc_raw = pd.read_csv('../data/raw/btc_usd_raw.csv', index_col='Date', parse_dates=True)

print("Features shape:", btc_scaled.shape)
btc_scaled.head()

Features shape: (2957, 6)


,log_returns,volatility_7d,volatility_21d,rsi,macd_diff,volume_norm
Date,,,,,,
2018-02-03,1.113386,1.895680,2.445072,-1.428217,-0.379480,-1.013379
2018-02-04,-3.075479,1.803715,2.603447,-1.706348,-0.468625,-1.030640
2018-02-05,-5.181169,2.702304,3.094989,-2.033703,-0.676550,-0.375841
2018-02-06,3.203808,3.975173,2.925583,-1.516760,-0.629225,1.043060
2018-02-07,-0.532652,3.870700,2.924037,-1.553866,-0.562311,-0.225839


### 3.1 Selecting optimal number of states

We test HMMs with 2 to 5 hidden states and compare using AIC and BIC. Lower values indicate a better balance between model fit and complexity.

In [6]:
x = btc_scaled.values
n_range = range(2,6)
models = {}
result = []

for n in n_range:
    best = -np.inf
    best_model= None

    for attempt in range(10):
        model = GaussianHMM(n_components=n,covariance_type='full',n_iter=200,random_state=attempt)
        try:
            model.fit(x)
            score = model.score(x)
            if score> best:
                best = score
                best_model= model
        except:
            continue

    models[n]= best_model
    aic = -2 * best + 2 * (n * n + n * 6 * 2)
    bic = -2 * best + np.log(len(x)) * (n * n + n * 6 * 2)
    result.append({'n_states': n, 'log_likelihood': best, 'AIC': aic, 'BIC': bic})
    print(f"States={n}: Log-Likelihood={best:.2f}, AIC={aic:.2f}, BIC={bic:.2f}")

results_df = pd.DataFrame(result)

States=2: Log-Likelihood=-20488.75, AIC=41033.50, BIC=41201.27
States=3: Log-Likelihood=-18979.54, AIC=38049.07, BIC=38318.71
States=4: Log-Likelihood=-18006.44, AIC=36140.88, BIC=36524.36
States=5: Log-Likelihood=-17323.62, AIC=34817.25, BIC=35326.56


In [ ]:

for metric in ['AIC', 'BIC']:
    print(f"\n--- {metric} ---")
    values = results_df[metric].values
    for i in range(1, len(values)):
        improvement = values[i-1] - values[i]
        pct = (improvement / abs(values[i-1])) * 100
        print(f"  {i+1}→{i+2} states: improvement = {improvement:.2f} ({pct:.1f}%)")


--- AIC ---
  2→3 states: improvement = 2984.43 (7.3%)
  3→4 states: improvement = 1908.19 (5.0%)
  4→5 states: improvement = 1323.63 (3.7%)

--- BIC ---
  2→3 states: improvement = 2882.56 (7.0%)
  3→4 states: improvement = 1794.34 (4.7%)
  4→5 states: improvement = 1197.80 (3.3%)
